In [ ]:
# ======================================================
# Notebook: 4D Black-box Optimisation (SVM surrogate)
# Inputs: (20,4) | Output: (20,)
# Goal: maximise yield
# ======================================================

import numpy as np
from sklearn.svm import SVR

# Load data
X = np.load("/mnt/data/initial_inputs.npy")      # (20,4)
y = np.load("/mnt/data/initial_outputs.npy")     # (20,)

# Train SVM regression surrogate
model = SVR(kernel="rbf", C=10.0, gamma="scale")
model.fit(X, y)

# Generate candidate samples within bounds
bounds = [(X[:,i].min(), X[:,i].max()) for i in range(4)]
num_candidates = 5000
X_grid = np.column_stack([
    np.random.uniform(b[0], b[1], num_candidates) for b in bounds
])

# Predict yield
preds = model.predict(X_grid)

# Exploration term (distance from existing samples)
dist = np.min(np.linalg.norm(X_grid[:,None,:] - X[None,:,:], axis=2), axis=1)

# Acquisition (exploit + explore)
acquisition = preds + 0.1 * dist

# Select next (10,4) candidates
top_idx = np.argsort(acquisition)[-10:]
next_points = X_grid[top_idx]

print("Next (10,4) candidate inputs:")
print(next_points)